# 4 — Label the mutant: two dyes, two reactions, one file OpenFF reads

Notebook 3 turned wild-type fibronectin FN7–10 into the labeling construct,
p-azido-L-phenylalanine (AzF) at 1381 and cysteine at 1500. This notebook puts
the FRET pair on it. The donor carries a strained alkyne (a DBCO group) and goes
onto the azide by a copper-free click reaction, which closes a triazole ring. The
acceptor carries a maleimide and goes onto the cysteine thiol by a Michael
addition, which moves the thiol hydrogen onto the ring.

Neither reaction is "replace one hydrogen with one bond", so neither can be
written with `attach`'s leaving-atom keywords. Both are written as a **reaction
string**, an RDKit reaction SMARTS that `attach` reads as a rule. The last part
of the notebook does the step that makes the result useful: it describes the two
new residues to OpenFF Pablo with Pablo's own public API, from the atoms and
bonds mBuild holds, so that the written PDB file loads as a complete chemical
graph.

Everything below uses the public API of mBuild, RDKit, the OpenFF Toolkit and
Pablo. There is no helper that hides a step.

In [ ]:
import logging
import time
import warnings

warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

from rdkit import Chem
from rdkit.Chem import Draw

from mbuild.biopolymers import REACTIONS, Protein, prepare_fragment
from demo_utils import show_protein

logging.getLogger("mbuild").setLevel(logging.WARNING)

protein = Protein("1fnf_S1381AzF_S1500C.pdb", download=True)
azf, cys = protein.get_residue(1381, chain_id="A"), protein.get_residue(1500, chain_id="A")
print(f"{protein.n_particles} atoms, net formal charge {protein.net_formal_charge}")
print(f"{azf.name} {azf.resnum}: charges {azf.atom_formal_charges}   |   {cys.name} {cys.resnum}: {sorted(p.name for p in cys.particles())}")

## The two dyes

Both SMILES come from the paper, with two edits made here. The acceptor as
published is the NHS-ester form, which reacts with amines; the paper used the
**maleimide-activated** form on the cysteine, so the NHS ester is replaced by an
amide to N-(2-aminoethyl)maleimide. And both dyes were drawn with three of their
four sulfonates as acids; at pH 7 all four are anions, so they are written
deprotonated and each dye carries a charge of −3. The formal charges in a SMILES
are what mBuild stores and what the force field will see, so this is where the
protonation state is decided.

`prepare_fragment` turns each SMILES into a `Residue` with unique atom names and
the formal charges. No `*` is needed: the reaction template picks the atoms.

In [ ]:
DONOR = ("CC1(CCCCCC(=O)NCCC(=O)N2Cc3ccccc3C#Cc3ccccc32)C(/C=C/C=C2/N(CCCS(=O)(=O)[O-])"
         "c3ccc(S(=O)(=O)[O-])cc3C2(C)C)=[N+](CCCS(=O)(=O)[O-])c2ccc(S(=O)(=O)[O-])cc21")
ACCEPTOR = ("CC1(C)C(/C=C/C=C/C=C2\\N(CCCS(=O)(=O)[O-])c3ccc(S(=O)(=O)[O-])cc3C2(C)CCCCC(=O)NCCN2C(=O)C=CC2=O)"
            "=[N+](CCCS(=O)(=O)[O-])c2ccc(S(=O)(=O)[O-])cc21")

donor = prepare_fragment(DONOR, "DON")
acceptor = prepare_fragment(ACCEPTOR, "ACC")
for dye in (donor, acceptor):
    print(f"{dye.name}: {dye.n_particles} atoms, formal charge {dye.formal_charge}")
Draw.MolsToGridImage([Chem.MolFromSmiles(DONOR), Chem.MolFromSmiles(ACCEPTOR)], molsPerRow=2,
                     subImgSize=(520, 300), legends=["DON: donor with a DBCO alkyne", "ACC: acceptor with a maleimide"])

## The reaction strings

`mbuild.biopolymers.REACTIONS` ships a few validated strings. Each has two
reactant templates, the protein side first, and one product template. Atoms
without a map number leave; bonds that appear in the product form; bonds whose
order changes take the product's order; charges the product states are applied.

In [ ]:
for name in ("azide-alkyne triazole", "thiol-maleimide"):
    print(f"{name:24s} {REACTIONS[name]}")

## Click the donor onto AzF

The triazole joins the two sides by **two** bonds, so this reaction is a ring
closure. `attach` places the dye by fitting it to both bonds at once and then
relaxes it with the protein held fixed until both are at bond length.

One more thing is decided here. A residue library such as Pablo's describes a
bond between residues as a *crosslink*, and it allows one crosslink per residue.
Two bonds between the same two residues cannot be declared. The product is
therefore written as **one residue**: `merge=True` moves the dye's atoms into
residue 1381, which keeps its number, takes the dye's charges, and drops its CCD
template because no CCD component describes it. It is then renamed, since the
name `4II` would tell a library to expect the azide.

In [ ]:
start = time.time()
labeled = protein.attach(donor, resnum=1381, atom_name="N3", chain_id="A",
                         reaction="azide-alkyne triazole", merge=True)
labeled.name = "TZ1"
print(f"{time.time() - start:.0f} s")
print(f"{labeled.name} {labeled.resnum}: {labeled.n_particles} atoms, formal charge {labeled.formal_charge}, "
      f"template {labeled.template}, azide charges left: {[n for n in ('N2', 'N3') if n in labeled.atom_formal_charges] or 'none'}")
print("residues:", len(list(protein.residues())), "| bonds recorded by this step:", len([b for b in protein.cross_bonds if b.reaction]))

## Add the acceptor to the cysteine

The thiol adds across the maleimide double bond. The sulfur bonds one carbon,
the thiol hydrogen ends up on the other, and the double bond becomes single.
Only one bond joins the two sides, so the dye stays its own residue and the bond
is recorded: `bond_records()` names the residues, the atoms, the hydrogen that
left the cysteine (it moved to the dye, but from the file's point of view it is
gone from the cysteine) and the reaction. That record is what a residue library
needs.

The warning that the next cell prints describes the rigid placement, before the relaxation that follows it inside `attach`: a 111-atom dye dropped onto a protein surface overlaps something, and the relaxation with the protein held fixed clears it.

In [ ]:
start = time.time()
record = protein.attach(acceptor, resnum=1500, atom_name="SG", chain_id="A", reaction="thiol-maleimide")
print(f"{time.time() - start:.0f} s")
bond = [r for r in protein.bond_records() if "reaction" in r][0]
print({k: v for k, v in bond.items() if k != "reaction"})
print(f"net formal charge of the labeled protein: {protein.net_formal_charge}   (−6 from the protein, −3 per dye)")

In [ ]:
show_protein(protein, link_selection="1381:A or 1500:A", fragment_selection="[TZ1] or [ACC]")

## Write the file

Pablo's cysteine definition already carries a crosslink, the disulfide, and it
takes only one, so the labeled cysteine also gets a name of its own. This is
the same renaming any force field asks for when it has its own name for a
modified residue.

In [ ]:
record.residue1.name = "CYL"
record.residue1.hetatm = True
protein.save_pdb("1fnf_labeled.pdb", overwrite=True)
print(open("1fnf_labeled.pdb").read().count("\n"), "lines written")

## Describe the new residues to OpenFF Pablo

Pablo reads a PDB file by matching every residue to a *residue definition*: the
free molecule with its atom names, elements, bonds, bond orders and formal
charges, plus the atoms that leave when the residue bonds to a neighbour. The
CCD supplies those for the standard residues. For `TZ1`, `CYL` and `ACC` we
build them ourselves, and the source is the mBuild residue, which holds exactly
that information.

The function below is the whole recipe:

1. Copy the residue's atoms, bonds, bond orders and formal charges into an RDKit
   molecule.
2. A residue in the chain is described as the free amino acid, so add the atoms a
   peptide bond removes: a second hydrogen on `N`, and `OXT` with its `HXT` on
   `C`. Mark them as leaving atoms, and declare Pablo's `PEPTIDE_BOND`.
3. An atom that bonds another residue by a crosslink gets a hydrogen that
   completes its valence in the free molecule. It is a bookkeeping atom, not a
   reaction intermediate: Pablo's own cysteine describes the disulfide the same
   way, through `HG`. `with_crosslink` below marks it as the atom that leaves.
4. Convert to an OpenFF `Molecule`, transfer the atom names, and call
   `ResidueDefinition.from_molecule`.

In [ ]:
from openff.pablo import STD_CCD_CACHE, ResidueDefinition, topology_from_pdb
from openff.pablo.chem import PEPTIDE_BOND
from openff.toolkit import Molecule

BOND_TYPES = {1.0: Chem.BondType.SINGLE, 2.0: Chem.BondType.DOUBLE, 3.0: Chem.BondType.TRIPLE, 1.5: Chem.BondType.AROMATIC}


def definition_from_residue(residue, name, in_chain, crosslink_atoms=()):
    '''Build a Pablo ResidueDefinition from an mBuild Residue.

    in_chain adds the peptide leaving atoms and declares the peptide bond.
    crosslink_atoms are atoms bonded to another residue; each gets a
    hydrogen that with_crosslink() will mark as leaving. Returns the
    definition and the names of those hydrogens.
    '''
    editable, index, names = Chem.RWMol(), {}, []
    for particle in residue.particles():
        atom = Chem.Atom(particle.element.symbol)
        atom.SetFormalCharge(residue.atom_formal_charges.get(particle.name, 0))
        atom.SetNoImplicit(True)
        index[particle] = editable.AddAtom(atom)
        names.append((particle.name, False))
    for p1, p2, data in residue.bonds(return_bond_order=True):
        editable.AddBond(index[p1], index[p2], BOND_TYPES[float(data["bond_order"])])

    taken = {n for n, _ in names}

    def fresh(name):
        while name in taken:
            name += "X"
        taken.add(name)
        return name

    def add(symbol, to, name, leaving):
        atom = Chem.Atom(symbol)
        atom.SetNoImplicit(True)
        new = editable.AddAtom(atom)
        editable.AddBond(to, new, Chem.BondType.SINGLE)
        names.append((fresh(name), leaving))
        return new

    if in_chain:
        add("H", index[next(residue.particles_by_name("N"))], "H2", leaving=True)
        oxt = add("O", index[next(residue.particles_by_name("C"))], "OXT", leaving=True)
        add("H", oxt, "HXT", leaving=True)
    caps = []
    for atom_name in crosslink_atoms:
        add("H", index[next(residue.particles_by_name(atom_name))], "H" + atom_name[1:], leaving=False)
        caps.append(names[-1][0])

    mol = editable.GetMol()
    Chem.SanitizeMol(mol)
    molecule = Molecule.from_rdkit(mol, allow_undefined_stereo=True)
    for atom, (atom_name, leaving) in zip(molecule.atoms, names):
        atom.name = atom_name
        atom.metadata["leaving_atom"] = leaving
    definition = ResidueDefinition.from_molecule(
        molecule, residue_name=name, linking_bond=PEPTIDE_BOND if in_chain else None
    )
    return definition, caps

The click product `TZ1` sits in the chain and has no crosslink. The labeled
cysteine `CYL` sits in the chain and crosslinks through `SG`; the dye `ACC`
is a free residue that crosslinks through the carbon the bond record names.
The crosslink declaration is then read straight from the record: the two
residues, the two atoms, and the atoms that are absent from the file because
the bond exists. On the cysteine side that is `HG`, as the record says; on the
dye side it is the bookkeeping hydrogen.

In [ ]:
tz1, _ = definition_from_residue(labeled, "TZ1", in_chain=True)
cyl, cyl_caps = definition_from_residue(record.residue1, "CYL", in_chain=True, crosslink_atoms=[bond["atom_names"][0]])
acc, acc_caps = definition_from_residue(record.residue2, "ACC", in_chain=False, crosslink_atoms=[bond["atom_names"][1]])
for d in (tz1, cyl, acc):
    print(f"{d.residue_name}: {len(d.atoms)} atoms, {sum(a.leaving for a in d.atoms)} leaving, "
          f"charge {sum(a.charge for a in d.atoms)}, linking bond {'yes' if d.linking_bond else 'no'}")

assert bond["leaving_atoms"][0] == cyl_caps, (bond["leaving_atoms"][0], cyl_caps)
library = STD_CCD_CACHE.with_({"TZ1": [tz1], "CYL": [cyl], "ACC": [acc]})
library = library.with_crosslink(
    residues=("CYL", "ACC"),
    linking_atoms=tuple(bond["atom_names"]),
    leaving_atoms=(cyl_caps, acc_caps),
    bond_order=bond["bond_order"],
)

## Load the file with Pablo

If Pablo can explain every atom and every bond from the definitions, it returns
an OpenFF `Topology`. Anything it cannot explain is an error naming the residue,
so a success here means the chemistry mBuild built survived the trip through
the file.

In [ ]:
topology = topology_from_pdb("1fnf_labeled.pdb", residue_library=library)
print(f"Pablo: {topology.n_atoms} atoms, {topology.n_molecules} molecule, {topology.n_bonds} bonds, "
      f"net charge {sum(a.formal_charge.m for a in topology.atoms)}")
print(f"mBuild: {protein.n_particles} atoms, {protein.n_bonds} bonds, net charge {protein.net_formal_charge}")
for name in ("TZ1", "CYL", "ACC"):
    atoms = [a for a in topology.molecule(0).atoms if a.metadata["residue_name"] == name]
    print(f"   {name}: {len(atoms)} atoms in the topology")

From here the path is notebook 2's: assign partial charges (the standard
residues from a library, the labeled residues from NAGL on a capped model of
each site), build an Interchange with the ff14SB port and Sage, and simulate.
The dyes are large conjugated anions, so the charge step deserves more care
than a demo gives it; nothing about the topology stands in the way.

**What to take away.** A reaction string extends `attach` to any reaction that
forms bonds, breaks bonds, changes orders and charges, or moves hydrogens,
without a new API per reaction. A product that must be one residue for a
downstream library is merged into the site residue and renamed. And a residue
definition for a downstream library is not a second description to maintain: it
is read off the mBuild residue, which already holds every atom, bond, order and
charge the library asks for.